# Week 4 — Supervised Learning Model Implementation

**Project:** Netflix Movies and TV Shows — Content Type Classification  
**Student:** Vikhyath Bharadwaj K S

This notebook predicts the catalogue field `type` (Movie or TV Show) from release year, content rating, primary genre, primary country, addition year/month, and counts of listed genres/countries. It intentionally excludes identifiers, title, description, director/cast, raw duration, and the target itself. This is a metadata classification exercise, not a model of viewing behavior or Netflix decisions.

The workflow uses a stratified held-out test set and three-fold stratified cross-validation on training data. Each preprocessing step is fitted within a scikit-learn pipeline to prevent leakage.

## 1. Imports and reproducibility
The source script contains the reusable modeling workflow and writes figures and actual metrics to the repository. The following cell imports the functions used for dataset inspection and later analysis.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = Path.cwd().parent
if str(ROOT / 'src') not in sys.path: sys.path.insert(0, str(ROOT / 'src'))
from week4_supervised_learning import RAW, FEATURES, prepare_frame, build_pipeline, run_analysis
print('Project root:', ROOT)
print('Source CSV:', RAW)

Project root: C:\Users\vikhy\OneDrive\Documents\Desktop\yuva-intern\yuva-intern-week1-netflix
Source CSV: C:\Users\vikhy\OneDrive\Documents\Desktop\yuva-intern\yuva-intern-week1-netflix\dataset\original\netflix_titles.csv


## 2. Dataset shape, types, data quality, and target feasibility
The supplied Kaggle snapshot is inspected directly. Exact duplicate rows and missing/invalid target rows are counted before filtering. Week 1's processed file is not blindly reused because it includes `is_movie`, `target_audience` (derived from rating), and duration features that could leak the answer or constrain comparisons. The raw source gives a transparent target and feature audit.

In [1]:
raw = pd.read_csv(RAW)
print('Shape:', raw.shape)
display(raw.dtypes.rename('dtype').to_frame())
display(raw.isna().sum().rename('missing').to_frame())
print('Exact duplicate rows:', raw.duplicated().sum())
print('Target values including missing:')
display(raw['type'].value_counts(dropna=False).to_frame('count'))
X, y, audit = prepare_frame(raw)
print(json.dumps({k: audit[k] for k in ['raw_rows','raw_columns','exact_duplicate_rows','model_rows','excluded_rows','target_distribution','feature_missing_counts']}, indent=2))

Shape: (7787, 12)
              dtype
show_id         str
type            str
title           str
director        str
cast            str
country         str
date_added      str
release_year  int64
rating          str
duration        str
listed_in       str
description     str
              missing
show_id             0
type                0
title               0
director         2389
cast              718
country           507
date_added         10
release_year        0
rating              7
duration            0
listed_in           0
description         0
Exact duplicate rows: 0
Target values including missing:
         count
type          
Movie     5377
TV Show   2410
{
  "raw_rows": 7787,
  "raw_columns": 12,
  "exact_duplicate_rows": 0,
  "model_rows": 7787,
  "excluded_rows": 0,
  "target_distribution": {
    "Movie": 5377,
    "TV Show": 2410
  },
  "feature_missing_counts": {
    "rating": 0,
    "primary_genre": 0,
    "primary_country": 0,
    "release_year": 0,
    "year_ad

## 3. Candidate task selection
**Selected task: content type classification.** The dataset labels each catalogue title as Movie or TV Show and has a reasonably usable target for both groups. Content rating classification has missing and unevenly represented labels, with metadata features likely to encode rating conventions. Movie-duration regression is feasible only after restricting to movies and validating duration strings, so it answers a narrower question. Content type supports a more complete end-to-end classifier across the catalogue.

To avoid trivial target leakage, `duration` and engineered duration values are excluded because the units explicitly say minutes or seasons. `show_id`, title, target-derived fields, and descriptive free text are excluded as well.

## 4. Modeling-focused exploratory analysis
The target is moderately imbalanced, so accuracy alone is insufficient; macro F1 and per-class precision/recall are also reported. Release year, genre, rating and country may differ by content type. The charts are generated from the full feature table for descriptive EDA only; all fitted preprocessing, model comparison and tuning use training folds.

In [1]:
summary = run_analysis()
print('Target distribution:', summary['target_distribution'])
print('Feature missing counts:', summary['feature_missing_counts'])
print('Saved visualizations to:', ROOT / 'visualizations' / 'week4')

Raw rows: 7787; modeling rows: 7787; features: 8
              model  cv_macro_f1_mean  test_accuracy  test_macro_f1  test_weighted_f1
Logistic Regression          0.992326       0.995507       0.994734          0.995503
      Decision Tree          0.992111       0.994865       0.993971          0.994856
      Random Forest          0.992521       0.994223       0.993260          0.994232
K-Nearest Neighbors          0.954866       0.960205       0.952367          0.959734
         Linear SVM          0.994013       0.996149       0.995504          0.996153
   Tuned Linear SVM          0.994384       0.996791       0.996251          0.996793
Best parameters: {'model__C': 2.0, 'model__class_weight': 'balanced'}
Classification report:
               precision    recall  f1-score   support

       Movie       1.00      1.00      1.00      1076
     TV Show       0.99      1.00      0.99       482

    accuracy                           1.00      1558
   macro avg       1.00      1.00    

## 5. Feature matrix and leakage-safe preprocessing
The primary genre and country are the first listed values; corresponding counts retain a small amount of multi-valued metadata without exploding the row count. Numeric fields use median imputation followed by standard scaling. Categorical fields use most-frequent imputation and one-hot encoding; categories observed fewer than 20 times are pooled as infrequent, and unseen categories are handled safely. All steps live inside a `ColumnTransformer` and `Pipeline`, so fitting occurs independently inside each training fold.

In [1]:
display(X.head())
print('Feature set:', FEATURES)
print('No target or identifier in features:', 'type' not in FEATURES and 'show_id' not in FEATURES)
print('Missing target after filtering:', y.isna().sum())

  rating           primary_genre primary_country  release_year  year_added  month_added  genre_count  country_count
0  TV-MA  International TV Shows          Brazil          2020      2020.0          8.0            3              1
1  TV-MA                  Dramas          Mexico          2016      2016.0         12.0            2              1
2      R           Horror Movies       Singapore          2011      2018.0         12.0            2              1
3  PG-13      Action & Adventure   United States          2009      2017.0         11.0            3              1
4  PG-13                  Dramas   United States          2008      2020.0          1.0            1              1
Feature set: ['rating', 'primary_genre', 'primary_country', 'release_year', 'year_added', 'month_added', 'genre_count', 'country_count']
No target or identifier in features: True
Missing target after filtering: 0


## 6. Split, cross-validation, and candidate estimators
An 80/20 train-test split is stratified with random state 42. Five algorithms are compared on the training set using three-fold shuffled stratified CV: logistic regression, decision tree, random forest, k-nearest neighbors, and linear SVM. The test set remains outside model selection and hyperparameter tuning. The highest-CV macro-F1 family is then tuned over `C` and class-weight settings by an eight-configuration grid (4 × 2) under the same three-fold CV, optimizing macro F1.

In [1]:
results = json.loads((ROOT / 'dataset' / 'processed' / 'week4_model_results.json').read_text(encoding='utf-8'))
display(pd.DataFrame(results['comparison'])[['model','cv_accuracy_mean','cv_accuracy_std','cv_macro_f1_mean','cv_macro_f1_std','test_accuracy','test_macro_f1','test_weighted_f1']].round(4))
print('Split:', results['split'])
print('Best parameters:', results['final_model']['best_params'])

                 model  cv_accuracy_mean  cv_accuracy_std  cv_macro_f1_mean  cv_macro_f1_std  test_accuracy  test_macro_f1  test_weighted_f1
0  Logistic Regression            0.9934           0.0002            0.9923           0.0003         0.9955         0.9947            0.9955
1        Decision Tree            0.9933           0.0014            0.9921           0.0017         0.9949         0.9940            0.9949
2        Random Forest            0.9936           0.0006            0.9925           0.0007         0.9942         0.9933            0.9942
3  K-Nearest Neighbors            0.9620           0.0018            0.9549           0.0021         0.9602         0.9524            0.9597
4           Linear SVM            0.9949           0.0002            0.9940           0.0003         0.9961         0.9955            0.9962
5     Tuned Linear SVM            0.9952           0.0004            0.9944           0.0005         0.9968         0.9963            0.9968
Split: {'trai

## 7. Held-out evaluation and interpretation
The primary score is macro F1, giving equal weight to Movie and TV Show. Accuracy and weighted F1 provide additional context. The confusion matrix and classification report show class-level errors. Linear SVM coefficients summarize conditional associations in the fitted linear model; they are not causal effects and can be unstable when categories are correlated.

In [1]:
display(pd.DataFrame(results['final_model']['classification_report']).T.round(4))
display(pd.DataFrame(results['final_model']['confusion_matrix'], index=results['final_model']['labels'], columns=results['final_model']['labels']))
display(pd.DataFrame(results['top_features_abs_coefficient']))

              precision  recall  f1-score    support
Movie            0.9991  0.9963    0.9977  1076.0000
TV Show          0.9918  0.9979    0.9948   482.0000
accuracy         0.9968  0.9968    0.9968     0.9968
macro avg        0.9954  0.9971    0.9963  1558.0000
weighted avg     0.9968  0.9968    0.9968  1558.0000
         Movie  TV Show
Movie     1072        4
TV Show      1      481
                                                feature  mean_absolute_coefficient
0           categorical__primary_genre_British TV Shows                   1.915995
1               categorical__primary_genre_Anime Series                   1.843868
2                 categorical__primary_genre_Docuseries                   1.801142
3     categorical__primary_genre_International TV Shows                   1.786912
4                     categorical__primary_genre_Dramas                   1.684501
5                   categorical__primary_genre_Kids' TV                   1.669815
6                   categoric

## 8. Visual evaluation
Each figure is stored in `visualizations/week4/` and embedded below. The target and feature distribution plots describe the snapshot. The model-comparison bar plot uses the final held-out test set only for reporting; do not use it to tune further.

### Target class distribution
![Target class distribution](../visualizations/week4/w4_01_target_distribution.png)

### Release year by content type
![Release year by content type](../visualizations/week4/w4_02_release_year_by_type.png)

### Most common primary genres
![Most common primary genres](../visualizations/week4/w4_03_genre_by_type.png)

### Held-out macro F1 model comparison
![Held-out macro F1 model comparison](../visualizations/week4/w4_05_model_comparison.png)

### Tuned model confusion matrix
![Tuned model confusion matrix](../visualizations/week4/w4_04_confusion_matrix.png)

## 9. Results and limitations
This dataset is a static catalogue snapshot with no user interactions, view counts, budgets, critic scores, or title availability histories. Even a strong classifier would show metadata associations rather than audience preference or business outcomes. Random splitting can place titles from the same creators/franchises across partitions; no temporal or group-based generalization claim is made. Macro F1, class-level recall, the held-out confusion matrix, and the actual cross-validation spread should be considered together.

In [1]:
print('Selected task: content type classification')
print('Final metrics:', {k: results['final_model'][k] for k in ['test_accuracy','test_macro_f1','test_weighted_f1','test_precision_macro','test_recall_macro']})
print('Interpretation:', results['interpretation'])

Selected task: content type classification
Final metrics: {'test_accuracy': 0.996790757381258, 'test_macro_f1': 0.9962513528092959, 'test_weighted_f1': 0.996793495475532, 'test_precision_macro': 0.9954103054351899, 'test_recall_macro': 0.9971039195421803}
Interpretation: The model predicts a catalogue metadata label. It does not predict viewer preference, popularity, quality, recommendations or future Netflix decisions.


## 10. Reproducibility record
The complete workflow is implemented in `src/week4_supervised_learning.py`; figures, metrics, split details, and coefficient summaries are saved under `visualizations/week4/` and `dataset/processed/week4_model_results.json`. Run `python src/week4_supervised_learning.py` from the repository root to regenerate them. The report is built from the saved actual evaluation results.